# 제공 데이터 탐색 (EDA)

가이드 4-4 탐색 항목: 구성 · 기간 · 길이 · 품질 · 내용

대상: `data/raw/02_AI반도체_묶음_*.csv` (키워드별 파일 2개의 합본)

In [1]:
import glob, re
import pandas as pd

# 프로젝트 루트 기준 경로 (notebooks/ 에서 실행)
f = glob.glob('../data/raw/*.csv')[0]
df = pd.read_csv(f, encoding='utf-8-sig')
df['본문전문'] = df['본문전문'].fillna('')
df['d'] = pd.to_datetime(df['발행일'])
print(df.shape)
df.head(3)

(120057, 8)


,제목,본문전문,원문URL,발행일,매체명,문서종류,수집키워드,d
0,"국산 AI 칩·보드, 서울대 교육 과정에 활용",서울대가 국산 인공지능(AI) 반도체 칩과 보드를 고급 AI 반도체 인재 양성에 활...,https://n.news.naver.com/mnews/article/030/000...,2023-12-19T07:23:00,전자신문,news,AI 반도체; 인공지능 반도체,2023-12-19 07:23:00
1,"KISDI, AI 반도체 시장 현황 보고서 발간",정보통신정책연구원(KISDI)은 AI 반도체 시장 현황과 전망을 담은 보고서를 발간...,https://n.news.naver.com/mnews/article/092/000...,2024-08-12T12:08:00,지디넷코리아,news,AI 반도체,2024-08-12 12:08:00
2,AI반도체커뮤니티 “새해 AI 반도체 설계 경진대회 추진”,새해 인공지능(AI) 반도체 기술 확산과 인재 발굴을 위한 경진대회가 열린다. 국내...,https://n.news.naver.com/mnews/article/030/000...,2023-12-28T13:49:00,전자신문,news,AI 반도체; 인공지능 반도체,2023-12-28 13:49:00


## 1. 구성 — 결측, 문서 종류, 수집 키워드, 매체

In [2]:
print(df.isna().sum())
print(df['문서종류'].value_counts())
print(df['수집키워드'].value_counts())
print('매체 수:', df['매체명'].nunique())
df['매체명'].value_counts().head(15)

제목       0
본문전문     0
원문URL    0
발행일      0
매체명      0
문서종류     0
수집키워드    0
d        0
dtype: int64
문서종류
news    120057
Name: count, dtype: int64
수집키워드
AI 반도체              68863
AI 반도체; 인공지능 반도체    40330
인공지능 반도체            10864
Name: count, dtype: int64
매체 수: 99


매체명
이데일리      6472
뉴시스       6356
서울경제      5907
연합뉴스      5827
뉴스1       5466
파이낸셜뉴스    5405
한국경제      5260
머니투데이     4940
매일경제      4938
헤럴드경제     4742
아시아경제     4626
전자신문      4606
조선비즈      3045
디지털타임스    2639
데일리안      2562
Name: count, dtype: int64

## 2. 기간 — 연도·분기별 분포

In [3]:
print(df['d'].min(), df['d'].max())
print(df['d'].dt.year.value_counts().sort_index())
df.groupby(df['d'].dt.to_period('Q')).size()

2021-10-16 07:25:00 2026-10-16 15:16:00
d
2021      597
2022     4316
2023    10280
2024    24491
2025    32786
2026    47587
Name: count, dtype: int64


d
2021Q4      597
2022Q1      952
2022Q2     1267
2022Q3     1026
2022Q4     1071
2023Q1     1760
2023Q2     2613
2023Q3     2759
2023Q4     3148
2024Q1     5613
2024Q2     7207
2024Q3     6129
2024Q4     5542
2025Q1     6416
2025Q2     6573
2025Q3     7613
2025Q4    12184
2026Q1    10998
2026Q2    16208
2026Q3    17886
2026Q4     2495
Freq: Q-DEC, dtype: int64

## 3. 길이 — 본문 길이 분포, 과소·과다 문서

In [4]:
L = df['본문전문'].str.len()
print(L.describe(percentiles=[.05,.25,.5,.75,.95]))
print('200자 미만:', (L<200).sum(), '/ 5000자 초과:', (L>5000).sum())

count    120057.000000
mean       1574.583298
std        1187.011378
min           0.000000
5%          559.000000
25%         949.000000
50%        1322.000000
75%        1856.000000
95%        3318.000000
max       49137.000000
Name: 본문전문, dtype: float64
200자 미만: 1007 / 5000자 초과: 1860


## 4. 품질 — 중복, 보일러플레이트, 관련성

In [5]:
# 중복 추정 (정밀 판별은 indexer/dedup.py)
def norm_title(t):
    return re.sub(r'\[[^\]]*\]|\([^)]*\)|[^가-힣A-Za-z0-9]', '', str(t)).lower()
df['nt'] = df['제목'].map(norm_title)
df['head'] = df['본문전문'].str.replace(r'\s+', '', regex=True).str[:80]
print('URL 중복:', df['원문URL'].duplicated().sum())
print('본문 완전 중복:', df['본문전문'].duplicated().sum())
print('정규화 제목 or 본문 앞 80자 중복:', (df['nt'].duplicated() | df['head'].duplicated()).sum())

URL 중복: 4
본문 완전 중복: 1169
정규화 제목 or 본문 앞 80자 중복: 8374


In [6]:
# 기사 말미 보일러플레이트
for p in ['기사 섹션 분류 안내', '이 기사를 추천합니다', '기사를 더 보고 싶다면', '무단전재']:
    print(p, df['본문전문'].str.contains(p, regex=False).sum())

기사 섹션 분류 안내 9547


이 기사를 추천합니다 6857
기사를 더 보고 싶다면 6124


무단전재 3


In [7]:
# 관련성: 본문 내 핵심 키워드 등장 횟수
c = df['본문전문'].str.count(r'AI ?반도체|인공지능 ?반도체')
print(c.clip(upper=5).value_counts().sort_index())
t = df['제목'].str.contains('반도체|HBM|NPU|칩|AI 가속기|퓨리오사|리벨리온|딥엑스|모빌린트')  # collector/load_provided.py 와 동일
rel = t | (c >= 2)
print('관련성 필터:', rel.sum())
print('+ 2024 이후:', (rel & (df['d'] >= '2024-01-01')).sum())
print('+ 2025 이후:', (rel & (df['d'] >= '2025-01-01')).sum())

본문전문
0    60224
1    36372
2    10838
3     4817
4     2471
5     5335
Name: count, dtype: int64
관련성 필터: 47514
+ 2024 이후: 40262
+ 2025 이후: 29162


In [8]:
# 대표 질문 관련 키워드 커버리지
kw = {'HBM':'HBM','NPU':'NPU','온디바이스':'온디바이스','가속기':'가속기','삼성전자':'삼성전자',
      'SK하이닉스':'SK하이닉스','엔비디아':'엔비디아','한미반도체':'한미반도체','보조금':'보조금',
      '세액공제':'세액공제','수출규제':'수출 ?규제|수출통제','반도체특별법':'반도체 ?특별법','산업연구원':'산업연구원'}
pd.Series({k: df['본문전문'].str.contains(p, regex=True).sum() for k, p in kw.items()})

HBM       21452
NPU        6950
온디바이스      4216
가속기        6231
삼성전자      37112
SK하이닉스    29402
엔비디아      33681
한미반도체      2514
보조금        3654
세액공제       1499
수출규제       3274
반도체특별법     1185
산업연구원       799
dtype: int64

## 5. 내용 — 샘플 직접 읽기

In [9]:
for i in df.sample(3, random_state=7).index:
    print(df.at[i,'발행일'], df.at[i,'매체명'], df.at[i,'제목'])
    print(df.at[i,'본문전문'][:500], '\n---')

2025-06-10T05:55:00 뉴스1 '이재명 랠리' 사흘간 3.3조원 사들인 외국인…40%는 '여기' 쏠렸다
이재명 정부 출범 이후 외국인 투자자들의 자금이 국내 증시로 대거 몰렸다. 새 정부의 자본시장 활성화 정책 기대감이 커지면서 '허니문 랠리'는 사흘간 이어지고 있다. 외국인들은 3거래일 동안 3조 원 넘게 쓸어 담았다. 코스피 대표 종목인 반도체주부터 방산 금융 자동차주까지 매수세가 전방위로 퍼졌다. 정치적 불확실성 걷히자, 외인 3.3조 유입 정책 기대감↑ 10일 한국거래소에 따르면 외국인 투자자들은 지난 4일부터 9일까지 3거래일간 국내 주식 3조 3581억 원을 순매수했다. 반면 기관과 개인이 각각 1497억 원, 3조 972억 원 순매도했다. 기관은 지난 4~5일 순매수였지만, 이날 순매도 전환했다. 외국인 투자자들은 이재명 대통령 당선 직후인 지난 4일 1조 1995억 원을 순매수한 이후 5일 1조 196억 원, 6일 1조 1289억 원 등 매 거래일 1조 원 이상 순매수를 지속해 왔다. 증권가에서는 지난해 말 윤석열 전 대통령의 계엄령 사태 이후 이어졌던 정치적 불확실성이 
---
2023-03-02T06:12:00 아시아경제 [무역적자 1년]②반도체 수출악화…재고 10년만에 최대치
우리나라 최대 수출 품목인 반도체 부진 역시 무역수지 적자가 1년 연속 장기화하는 데 직접적인 영향을 끼쳤다. 특히 반도체 내 수출 비중이 큰 D램 및 낸드 등 메모리반도체 제품 가격 약세가 수출 감소의 직격탄이 됐다. 지난달 반도체 수출은 전년 대비 40% 이상 급감하며 지난해 8월 이후 7개월째 쪼그라들었다. 전문가들은 주요 반도체 제품가격 하락세가 올 상반기까지 지속할 것으로 예상하면서 무역수지 회복 역시 상당한 시간이 걸릴 수 있다고 내다봤다. 2일 산업통상자원부에 따르면 지난달 반도체 수출액이 59억6000만달러로 지난해 같은 기간 대비 42.5% 감소했다. 반도체 수출은 지난해 8월 7.8% 감소한 이후 감소 폭이 점차 확대했다. 40%대로 수출

## 정리 (1차)

- 120,057건, 본문 결측 86건, 문서 종류는 전부 news
- 2024년 이후 비중 87% (최근 편중)
- 본문에 'AI 반도체'가 0회 등장하는 문서 50% → 관련성 필터 필요
- 제목/본문 앞부분 기준 중복 약 8,400건 (재배포본 포함 시 증가 예상)
- 기사 말미 보일러플레이트 존재 → indexer/parse.py 에서 제거

## 6. 직접 수집 문서 탐색 — 공시·보도자료·보고서 (`store/docs.db`)

제공 뉴스만으로는 1차 근거가 없어 직접 수집한 문서도 같은 항목(구성·기간·길이·품질·내용)으로 탐색한다.
청킹 규칙(설정 B)은 여기서 확인한 문서 구조를 근거로 정했다 (`report/design.md 3부`).

In [10]:
import sqlite3
con = sqlite3.connect('file:../store/docs.db?mode=ro', uri=True)
docs = pd.read_sql('''SELECT d.doc_id, d.doc_type, d.source, d.title, substr(d.published_at,1,10) AS date,
                             length(d.body) AS raw_len, p.chars_after AS clean_len, p.status
                      FROM documents d LEFT JOIN parsed_documents p USING (doc_id)''', con)
docs['year'] = docs['date'].str[:4]
# 구성: 문서 종류별 건수와 발행처
print(docs['doc_type'].value_counts())
for t in ['dart', 'press', 'report']:
    print(f'\n[{t}] 발행처:', docs.loc[docs.doc_type == t, 'source'].value_counts().head(8).to_dict())

doc_type
news      47428
press       438
dart         33
report        7
Name: count, dtype: int64

[dart] 발행처: {'삼성전기': 5, 'SK하이닉스': 5, '한미반도체': 5, '삼성전자': 5, '이수페타시스': 5, '가온칩스': 4, '오픈엣지테크놀로지': 4}

[press] 발행처: {'산업통상자원부': 199, '산업통상부': 138, '과학기술정보통신부': 93, '기획재정부': 7, '재정경제부': 1}

[report] 발행처: {'산업연구원(KIET)': 7}


In [11]:
# 기간: 종류별 연도 분포
print(pd.crosstab(docs['doc_type'], docs['year']))

year      2021  2022  2023   2024   2025   2026
doc_type                                       
dart         0     5     7      7      7      7
news       197  1890  5165  11080  11730  17366
press        8    34    32     99    138    127
report       0     2     1      2      1      1


In [12]:
# 길이: 정제 후 본문 길이 (뉴스와 1차 자료의 규모 차이 → 청킹 방식을 문서 종류별로 다르게 한 근거)
print(docs.groupby('doc_type')['clean_len'].describe(percentiles=[.05, .5, .95]).round(0))

            count     mean      std      min       5%      50%       95%  \
doc_type                                                                   
dart         33.0  29181.0  10205.0  13497.0  14548.0  28224.0   42595.0   
news      47428.0   1537.0   1087.0      0.0    540.0   1312.0    3217.0   
press       438.0   9401.0  12449.0    326.0   1164.0   4260.0   34186.0   
report        7.0  55559.0  49966.0  12608.0  13041.0  38719.0  126022.0   

               max  
doc_type            
dart       44804.0  
news       27471.0  
press      96489.0  
report    128861.0  


In [13]:
# 품질: 정제 상태 (garbled = 글자 깨짐, empty = 본문 없음), 정제로 줄어든 비율
print(pd.crosstab(docs['doc_type'], docs['status'].fillna('없음')))
docs['removed_ratio'] = 1 - docs['clean_len'] / docs['raw_len']
print(docs.groupby('doc_type')['removed_ratio'].median().round(3))

status    empty  garbled     ok
doc_type                       
dart          0        0     33
news         10        0  47418
press         0        0    438
report        0        1      6
doc_type
dart      0.002
news      0.000
press     0.065
report    0.083
Name: removed_ratio, dtype: float64


In [14]:
# 내용: 종류별 샘플 직접 읽기 (구조 파악 → 청킹 경계 결정)
for t in ['dart', 'press', 'report']:
    did = docs[(docs.doc_type == t) & (docs.status == 'ok')].sample(1, random_state=3)['doc_id'].iloc[0]
    title, body = con.execute('SELECT d.title, p.body FROM documents d JOIN parsed_documents p USING (doc_id) WHERE doc_id=?', (did,)).fetchone()
    print(f'===== [{t}] {title}\n{body[:600]}\n')

===== [dart] 삼성전기 사업보고서 (2021.12) - II. 사업의 내용
## II. 사업의 내용

## 1. 사업의 개요

당사는 수동소자(MLCC, 칩인덕터, 칩처항)를 생산하는 컴포넌트 사업부문, 카메라모듈ㆍ통신모듈을 생산하는 광학통신솔루션 사업부문, 반도체패키지기판을 생산하는 패키지솔루션 사업부문의 총3개 사업부문으로 구성되어 있습니다.
지역별로는 수원에 위치한 본사를 포함하여 국내에 총 3개의 생산기지(수원, 세종, 부산)와 해외 총 7개의 생산기지(중국 4개, 태국, 필리핀, 베트남)를 보유,

15개의

자회사

와 1개의 손자회사를

두고 사업을 운영하고 있습니다.

2021년말 누적 매출액

은

전년 동기 대비 24.8% 상승한

9조 6,75

0

억원으로

, 각 사업부문별 매출액 및 매출비중은 컴포넌트 사업부문

4조 7,718억원

(49.32%), 광학통신솔

루션 사업부문 3조 2,241억원(33.32%), 패키지솔루션 사업부문 1조 6,791억원(17.3

6%)입니

다.

매출 유형별 매

출실적은

제품/상품 9조 5,756억원, 용역 및 기타매출 994억원으로 구성되어 있습니다

. 당

사의 주요

매출처

는 삼

성전자와 그 종속기업 및 Xiaomi이며,

주

요 매출처에 대한 매출비

===== [press] 화합물 전력반도체 산업의 미래를 그리다
화합물 전력반도체 산업의 미래를 그리다
- 『AI 시대, 반도체 산업전략』을 구체화할 차세대 전력반도체 추진단 포럼 개최
- ’30년까지 화합물 전력반도체 기술자립률· 국내 생산비중 2배 목표
산업통상부(장관 김정관, 이하 산업부)는 지난 10일 발표한 「AI 시대, 반도체
산업전략」의 세부과제를 구체화하기 위해 산학연 관계자 100여명과 함께
12.17(수) 그랜드머큐어 임피리얼 팰리스 서울 강남에서 ‘차세대 전력반도체
포럼’을 개최하였다. 차세대 전력반도체는 Si (실리콘) 대비 고온· 고전압에서
효율이 높은 화합물 소재(SiC(탄화규소), GaN(질화갈

## 정리 (2차) — 직접 수집 문서

- 구성: 공시 33(7개사 × 4~5년), 보도자료 438(산업부 337 · 과기정통부 93 · 기재부 8), KIET 보고서 7
- 기간: 뉴스처럼 2024년 이후로 편중되지만 2022~2023년 보도자료·공시가 있어 시점 질문의 1차 근거가 된다
- 길이: 정제 후 본문 중앙값 — 뉴스 1,312자, 보도자료 4,260자, 공시 28,224자, 보고서 38,719자
  → 뉴스는 통째로 또는 문장 단위, 1차 자료는 소제목·개조식·쪽 경계로 나누는 **문서 종류별 청킹**(설정 B)의 근거
- 품질: 보고서 1편 글자 깨짐(garbled) → 제외, 빈 뉴스 10건 제외. 보도자료·보고서는 정제로 본문의 6.5%·8.3%(머리말·목차·쪽 번호)가 줄었다
- 구조: 공시는 `## 소제목` + `|` 구분 표, 보도자료는 □ ○ 개조식 + 숫자 표, 보고서는 쪽 단위
- 주의: PDF 표가 셀 단위로 풀려 짧은 줄이 많다(보도자료 줄의 44%가 4자 이하). 공시 일부(33건 중 5건)는 서식 태그 때문에 단어·숫자 중간에 줄바꿈이 남는다
- 원본 뉴스 기준 2024년 이후 87%, 관련성 필터 후(47,428건) 85%